<a href="https://colab.research.google.com/github/furbyhaxx/CLM/blob/main/notebooks/CLM_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CLM inference on Colab, without vLLM

**Contrastive Language Models** answer typed questions about a state (yes/no, choice, score)
and rank candidate actions. Each answer takes one embedding per new text from a frozen
**Qwen3-8B** encoder, a small projection head (20M params) and a dot product per candidate.

The repo's quickstart serves the encoder with `vllm serve --runner pooling`. This notebook runs
the encoder **in this process with Unsloth** instead. `clm.unsloth_embedder` loads Qwen3-8B with
`FastLanguageModel` and takes the final-norm hidden state of the last token, the same vector the
vLLM pooling server returns. There is no second server, no port and no vLLM install.

| GPU | encoder weights | fits |
|---|---|---|
| T4 (free, 15 GB) | 4-bit Unsloth dynamic quant (~6 GB) | yes |
| L4 / A100 | 16-bit, exactly what the heads were trained on | yes |

The released heads were trained on 16-bit embeddings. The 4-bit encoder gives close but not
identical vectors, so for exact numbers use a GPU with at least 20 GB.

Runtime → *Run all* on a free T4 works end to end.

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# The CLM repo: the package (heads, Engine, the in-process encoder) and the train / eval scripts.
# It is put on sys.path rather than pip-installed: `contrastive-lm` depends on vLLM, which these
# notebooks do not use.
import os, sys, subprocess
CLM_REPO = "https://github.com/furbyhaxx/CLM"
CLM_REF = "main"
if os.path.isfile(os.path.join("..", "src", "clm", "__init__.py")):   # opened from a clone's notebooks/
    CLM_DIR = os.path.abspath("..")
else:
    CLM_DIR = os.path.abspath("CLM")
    if not os.path.isdir(CLM_DIR):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", CLM_REF, CLM_REPO, CLM_DIR], check=True)
for sub in ("src", "train", "preprocessing"):
    sys.path.insert(0, os.path.join(CLM_DIR, sub))
print("CLM:", CLM_DIR)

In [ ]:
# @title GPU
import torch
gpu = torch.cuda.get_device_properties(0)
GPU_GB = gpu.total_memory / 2**30
print(f"GPU = {gpu.name}, {GPU_GB:.1f} GB, bf16 = {torch.cuda.is_bf16_supported()}")

### Load the encoder with Unsloth

`UnslothEmbedder` is a drop-in replacement for the HTTP `Embedder`, with the same LRU cache and
the same `embed(texts) -> (vectors, tokens)` contract. It:

* loads `Qwen/Qwen3-8B` through `FastLanguageModel`. `LOAD_IN_4BIT = None` picks 4-bit below
  20 GB of GPU memory and 16-bit above; Unsloth swaps in its pre-quantized
  `unsloth/Qwen3-8B-unsloth-bnb-4bit`, so the download is ~6 GB instead of 16 GB,
* drops the 151k-token `lm_head` (~1.2 GB) that an encoder never uses,
* tokenizes without special tokens and keeps the **last** `MAX_TOKENS` tokens, like vLLM's
  `truncate_prompt_tokens`,
* sorts texts by length and batches them by token budget, so a cold batch of mixed lengths pads
  very little.

In [ ]:
import unsloth  # import first, so Unsloth patches transformers before anything else loads it
from clm.unsloth_embedder import UnslothEmbedder

ENCODER = "Qwen/Qwen3-8B"   # or a LoRA adapter dir from CLM_Finetune.ipynb (part 3)
MAX_TOKENS = 2048           # states longer than this keep their last 2048 tokens
LOAD_IN_4BIT = None         # None = auto (4-bit on a T4); True / False to force

embedder = UnslothEmbedder(ENCODER, max_tokens=MAX_TOKENS, load_in_4bit=LOAD_IN_4BIT)
print(f"encoder on GPU: {torch.cuda.memory_allocated() / 2**30:.1f} GB")

### The engine: encoder + reference head

`clm.heads.download()` fetches the 75 MB reference head
[`Contrastive-LM/CLM-v0.1-8B`](https://huggingface.co/Contrastive-LM/CLM-v0.1-8B). `Engine` is the
same in-process engine that `clm-serve` wraps. It also reserves a small GPU **vector cache** for
states and actions it has already projected (2% of the GPU by default).

In [ ]:
from clm import Engine
from clm.heads import download

engine = Engine(embedder=embedder, checkpoint=download())
print([m["name"] for m in engine.models()], "| heads on", engine.device)

### Ask typed questions about a state

* `Noul`: probability that a statement is true.
* `Choice`: a distribution over named options; each option is embedded as its description.
* `Score`: an ordered rubric; `score` is the expected level.

In [ ]:
from clm import Choice, Noul, Score

state = "Customer: my invoice was charged twice and nobody answers the phone!"
questions = {
    "urgency": Noul(instructions="Is this urgent?"),
    "department": Choice(instructions="Which team should handle this?",
                         criteria={"billing": "Charges, invoices, refunds",
                                   "technical": "Bugs and outages"}),
    "frustration": Score(instructions="How frustrated is the customer?",
                         criteria=["Calm", "Frustrated", "Very angry"]),
}
r = engine.answer(state, questions)
a = r["answers"]
print("urgency     p(true) =", round(a["urgency"]["noul"], 4))
print("department  ", a["department"]["choice"], a["department"]["probabilities"])
print("frustration ", round(a["frustration"]["score"], 3), "(0..2)")
print("usage       ", r["usage"])

### Rank free-form candidates

`rank` is the underlying primitive: it scores each candidate against the state. You can use it
for best-of-N answers, tool routing or next moves.

In [ ]:
engine.rank("What causes tides on Earth?",
            ["The Moon's gravitational pull.", "Photosynthesis in plants.", "Because the Earth is round."])

### Throughput and the caches

An agent asks about a changing state but a mostly fixed set of actions. Every text is embedded
once, and both the embedding (LRU) and its projection (GPU vector cache) are reused afterwards.
The first call below pays the encoder; the repeat is a cache hit.

For bulk work, call `embedder.embed(all_texts)` once up front. The encoder then sorts and batches
everything in one pass, and later `answer` calls only hit the cache.

In [ ]:
import time

def timed(fn, n=1):
    torch.cuda.synchronize(); t = time.perf_counter()
    for _ in range(n):
        out = fn()
    torch.cuda.synchronize()
    return out, (time.perf_counter() - t) / n * 1000

fresh = f"Customer #{time.time_ns()}: the app crashes every time I upload a photo."
_, cold = timed(lambda: engine.answer(fresh, questions))
_, warm = timed(lambda: engine.answer(fresh, questions), n=20)
print(f"new state: {cold:.1f} ms | revisited state: {warm:.2f} ms")

# bulk: 256 new states in one sorted, token-batched encoder pass
states = [f"Ticket {i}: " + ("the dashboard is slow and " * (1 + i % 12)) + "please help." for i in range(256)]
_, ms = timed(lambda: embedder.embed(states))
print(f"bulk embed: {len(states) / (ms / 1000):.0f} states/s")
print("vector cache:", {k: v for k, v in engine.arena.stats().items() if k != "pools"} if engine.arena else "off")

### Serve the API and the playground from Colab

`clm.server.create_app(engine)` is the same FastAPI app `clm-serve` runs, so the TypeSafe-compatible
`POST /v1/systemone`, `POST /v1/rank` and the web playground all work here with the in-process
encoder. Outside a notebook, the same thing is `clm-serve --local-encoder`.

In [ ]:
%%capture
!pip install -q fastapi uvicorn

In [ ]:
import threading, uvicorn
from clm.server import create_app

PORT = 8700
server = uvicorn.Server(uvicorn.Config(create_app(engine), host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started:
    time.sleep(0.1)

from clm import CLMClient
client = CLMClient(base_url=f"http://127.0.0.1:{PORT}")
r = client.system_one(state=state, questions=questions)
print(r.answers["department"].choice, r.answers["department"].probabilities, f"{r.latency_ms} ms")

In [ ]:
# Open the playground in a new browser tab (Colab only)
try:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT, path="/")
except ImportError:
    print(f"playground: http://localhost:{PORT}/")

### Your own heads (and encoders)

Fine-tuned heads from `CLM_Finetune.ipynb` load next to the reference head. A head only makes
sense with the encoder it was trained against. For a head trained jointly with a LoRA-adapted
encoder (part 3 of that notebook), set `ENCODER` above to the saved adapter directory.

In [ ]:
import glob
if os.path.isdir(ENCODER):   # a LoRA encoder: its own head, saved next to it
    mine = {"lora": os.path.join(os.path.dirname(os.path.abspath(ENCODER)), "best_head.pt")}
else:                        # heads trained on the frozen encoder (DeepSWE heads are verifiers)
    mine = {os.path.basename(os.path.dirname(p)): p for p in glob.glob("runs/*/best_head.pt")
            if not any(k in p for k in ("deepswe", "lora"))}
if mine:
    engine = Engine(embedder=embedder, checkpoint=download(), models=mine)
    for name in mine:
        print(name, engine.answer(state, questions, model=name)["answers"]["department"])
else:
    print("no runs/*/best_head.pt yet; train one with CLM_Finetune.ipynb")

### Free the GPU

In [ ]:
server.should_exit = True
del engine, embedder
import gc; gc.collect(); torch.cuda.empty_cache()

Next: **CLM_Eval.ipynb** reproduces the DeepSWE best-of-4 result and scores the typed-decisions
benchmark with this encoder. **CLM_Finetune.ipynb** trains heads, and optionally a LoRA on the
encoder, with Unsloth.